## Download Text8 Model

In [ ]:
!wget -O dfm.pt "https://www.dropbox.com/scl/fi/rno9fq8mpjs2bdctz7o53/dfm.pt?rlkey=1ge1wxv14b4a46b730hbltwkg&dl=1"

## Load the Pre-Trained Model

In [1]:
import sys
sys.path.append('../')

import torch
import torch.nn.functional as F
from flow_model import GPT, GPTConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Text8 uses 27 characters (a-z and space). The mask token acts as the 28th token.
vocab_size = 28  
mask_token_id = 27

# Load the checkpoint
ckpt_path = 'dfm.pt'
checkpoint = torch.load(ckpt_path, map_location=device)

# Configure and instantiate the model
model_args = checkpoint['model_args']
model_args['vocab_size'] = vocab_size
gptconf = GPTConfig(**model_args)
model = GPT(gptconf)

# Strip out the DDP prefix ('_orig_mod.') if it exists
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

model.load_state_dict(state_dict)
model.eval()
model.to(device)
model = torch.compile(model)
print("Model successfully loaded!")

# Create a decoder mapping (0 = space, 1-26 = a-z, 27 = mask)
itos = {0: ' '}
for i in range(1, 27):
    itos[i] = chr(ord('a') + i - 1)
itos[mask_token_id] = 'X'

def decode(token_list):
    return ''.join([itos.get(int(idx), '?') for idx in token_list])

Using device: cuda
GPT config GPTConfig(block_size=256, vocab_size=28, n_layer=12, n_head=12, n_embd=768, dropout=0.0, bias=False, qk_layernorm=True, do_x1_sc=True, mask_token_id=27, proper_timestep_emb=False, d3pm_loss_weighting=False, d3pm_loss_weighting_maxT=1000)
number of parameters: 86.16M
Model successfully loaded!


## Define Sampling Discrete Flow Model

In [2]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm
from contextlib import nullcontext
# ----------------- CONFIGS -----------------
# batch_size = 256
block_size = 256
vocab_size = 28 # 27 chars + 1 mask token
mask_token_id = 27

# total_samples = 512
dt = 0.001
max_t = 0.98
noise = 15.0      # Updated noise schedule
x1_temp = 1.0     # Temperature for predicting clean text
do_x1_sc = True   # Self-conditioning is ON
# -------------------------------------------

device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
ptdtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
ctx = torch.amp.autocast(device_type=device_type, dtype=ptdtype) if device_type == 'cuda' else nullcontext()
def sample_discrete_flow(model, batch_size, block_size=block_size, dt=dt, max_t=max_t, noise=noise, x1_temp=x1_temp, do_x1_sc=do_x1_sc):
    total_samples = batch_size
    
    mask_one_hot = torch.zeros((vocab_size,), device=device)
    mask_one_hot[mask_token_id] = 1.0
    
    b_idx = torch.arange(batch_size, device=device).repeat_interleave(block_size)
    d_idx = torch.arange(block_size, device=device).repeat(batch_size)
    
    all_generated_samples = []
    
    with torch.no_grad():
        with ctx: # Use mixed precision
            num_batches = total_samples // batch_size
            for batch_num in range(num_batches):
                print(f"Generating batch {batch_num + 1}/{num_batches}...")
                
                samples = mask_token_id * torch.ones((batch_size, block_size), device=device, dtype=torch.long)
                if do_x1_sc:
                    x1_sc = mask_token_id * torch.ones_like(samples)
                
                t = 0.0
                
                # Calculate total steps for the progress bar
                total_steps = int(max_t / dt)
                
                with tqdm(total=total_steps, desc="Diffusion Steps") as pbar:
                    while t <= max_t:
                        if do_x1_sc:
                            logits = model._run_net(samples, t * torch.ones((batch_size,), device=device), x1=x1_sc)
                        else:
                            logits, _ = model(samples, t * torch.ones((batch_size,), device=device))
                        
                        pt_x1_probs = F.softmax(logits / x1_temp, dim=-1)
                        
                        if do_x1_sc:
                            x1_sc = torch.multinomial(pt_x1_probs.view(-1, vocab_size), num_samples=1).view(batch_size, block_size).long()
                        
                        sample_is_mask = (samples == mask_token_id).view(batch_size, block_size, 1).float()
                        
                        step_probs = dt * pt_x1_probs * ((1 + noise * t) / (1 - t)) 
                        step_probs = step_probs * sample_is_mask
                        
                        step_probs += dt * (1 - sample_is_mask) * mask_one_hot.view(1, 1, -1) * noise
                        step_probs = torch.clamp(step_probs, min=0.0, max=1.0)
                        
                        step_probs[b_idx, d_idx, samples.flatten()] = 0.0
                        step_probs[b_idx, d_idx, samples.flatten()] = 1.0 - torch.sum(step_probs, dim=-1).flatten()
                        step_probs = torch.clamp(step_probs, min=0.0, max=1.0)
                        
                        samples = torch.multinomial(step_probs.view(-1, vocab_size), num_samples=1).view(batch_size, block_size)
                        
                        t += dt
                        pbar.update(1)
                    
                sample_is_mask = (samples == mask_token_id).view(batch_size, block_size).float()
                if do_x1_sc:
                     logits = model._run_net(samples, t * torch.ones((batch_size,), device=device), x1=x1_sc)
                else:
                     logits, _ = model(samples, t * torch.ones((batch_size,), device=device))
                     
                samples = torch.argmax(logits, dim=-1).float() * sample_is_mask + samples.float() * (1 - sample_is_mask)
                all_generated_samples.append(samples.long().cpu().numpy())
            
    return np.concatenate(all_generated_samples, axis=0)

## Sample Sequences and Decode

In [10]:
# Generate
print("Generating 4 sequences...")
generated_tokens = sample_discrete_flow(model, batch_size=4, block_size=256, x1_temp=1.0)

# Print results
for i, row in enumerate(generated_tokens):
    text = decode(row)
    print(f"\n--- Sample {i+1} ---")
    print(text)

Generating 4 sequences...
Generating batch 1/1...


Diffusion Steps:   0%|          | 0/980 [00:00<?, ?it/s]


--- Sample 1 ---
en iseahon jim colb danny sheldon loe bruce hackasole l onni kettaken fathers rogar hans bert schmitten tintken inholt magnus fridder er stanij pennista kinghass jorianno dumareiro henrer thomatte jansen mith artani manuel barrera one nine four three novem

--- Sample 2 ---
dol sans dodesta caplo an grisco sacio feriolo sar triva el sacio t came away literatura tu literatura la foro de la campo que apor literatura y espa ola del vito noce auberal y quimaro ne ha iso el casuro non posa an premas fano en ilatre et un libera o u

--- Sample 3 ---
i a one zero men river one of the responses to trani kv nar riker is near siev sarajski suffixed by the river wise now merharted to open the second section of the lest dungilald position kilimij rv r and advance to kurzov arguing that the second defensive 

--- Sample 4 ---
rock the gibelon su two the chorus of the sheef the weak the stagnant descendant sabre tenor zoucha the princess five eutines the widgot the dagotak sages sc

## Sample and save data for evaluation

In [3]:
# Sample 128 sequence and save them to samples.txt
import os
sample_dir = 'samples'
os.makedirs(sample_dir, exist_ok=True)
for temp in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print(f"Generating samples with temperature {temp}...")
    samples_np = sample_discrete_flow(model, batch_size=512, block_size=256, x1_temp=temp)
    samples_path = os.path.join(sample_dir, f'samples_temp_{temp}.txt')
    with open(samples_path, 'w') as f:
        for sample_idx in range(samples_np.shape[0]):
            f.write(decode(samples_np[sample_idx]) + '\n')

Generating samples with temperature 0.5...
Generating batch 1/1...


Diffusion Steps:   0%|          | 0/980 [00:00<?, ?it/s]

Generating samples with temperature 0.6...
Generating batch 1/1...


Diffusion Steps:   0%|          | 0/980 [00:00<?, ?it/s]

Generating samples with temperature 0.7...
Generating batch 1/1...


...

Diffusion Steps:   0%|          | 0/980 [00:00<?, ?it/s]

Generating samples with temperature 0.9...
Generating batch 1/1...


Diffusion Steps:   0%|          | 0/980 [00:00<?, ?it/s]

Generating samples with temperature 1.0...
Generating batch 1/1...


Diffusion Steps:   0%|          | 0/980 [00:00<?, ?it/s]